In [16]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

In [17]:
# =============================================================================
# 1. 基本定义

In [18]:
# =============================================================================

folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")

SOC_ORDER = ["90%", "50%", "10%"]

CYCLE_START_ZUSTAND = "CHA"
CYCLE_START_CURRENT_LEVEL = 1.5
CURRENT_ROUND_DIGITS = 1

OUTPUT_COLUMNS = ["SOH", "SOC", "File", "Time", "Current_level", "Zustand"]

In [19]:
# =============================================================================
# 2. 读取数据

In [20]:
# =============================================================================

def natural_key(x):
    x = str(x)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", x)]


def get_soh_from_path(path):
    text = str(path)

    match = re.search(r"SOH[_\s-]*(\d+(?:\.\d+)?)", text, flags=re.IGNORECASE)
    if match:
        return match.group(1)

    match = re.search(r"(\d+(?:\.\d+)?)\s*SOH", text, flags=re.IGNORECASE)
    if match:
        return match.group(1)

    return pd.NA


parquet_files = sorted(folder_path.glob("*.parquet"), key=natural_key)

if not parquet_files:
    raise FileNotFoundError(f"No parquet files found in {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name

    if "SOH" not in temp.columns:
        temp["SOH"] = get_soh_from_path(file)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [21]:
# =============================================================================
# 3. 基础处理

In [22]:
# =============================================================================

df["Time"] = pd.to_datetime(df["Time"], utc=True, errors="coerce")
df["Current"] = pd.to_numeric(df["Current"], errors="coerce")
df["Zustand"] = df["Zustand"].astype(str).str.strip()

df = df.dropna(subset=["Time", "Current", "Zustand"]).copy()
df = df.sort_values("Time").reset_index(drop=True)

df["Current_level"] = df["Current"].round(CURRENT_ROUND_DIGITS)
df["Zustand_upper"] = df["Zustand"].str.upper()

In [23]:
# =============================================================================
# 4. 压缩连续段，生成识别序列

In [24]:
# =============================================================================

is_new_step = (
    df["Current_level"].ne(df["Current_level"].shift())
    | df["Zustand_upper"].ne(df["Zustand_upper"].shift())
)

sequence = df.loc[is_new_step, ["SOH", "File", "Time", "Current_level", "Zustand", "Zustand_upper"]].copy()
sequence = sequence.reset_index(drop=True)

In [25]:
# =============================================================================
# 5. 根据 +1.5A CHA 定义 cycle，并赋值 SOC

In [26]:
# =============================================================================

cycle_start_mask = (
    sequence["Zustand_upper"].eq(CYCLE_START_ZUSTAND)
    & sequence["Current_level"].eq(CYCLE_START_CURRENT_LEVEL)
)

cycle_start_positions = np.flatnonzero(cycle_start_mask.to_numpy())

if len(cycle_start_positions) == 0:
    raise ValueError("没有找到 +1.5A CHA 作为 cycle 起点。")

sequence["cycle_id"] = np.searchsorted(
    cycle_start_positions,
    np.arange(len(sequence)),
    side="right",
)

sequence = sequence[sequence["cycle_id"] > 0].copy()

sequence["SOC"] = sequence["cycle_id"].apply(
    lambda x: SOC_ORDER[(x - 1) % len(SOC_ORDER)]
)

In [27]:
# =============================================================================
# 6. 输出列

In [28]:
# =============================================================================

recognition_sequence = sequence[OUTPUT_COLUMNS].copy()

output_file = folder_path / "recognition_sequence_clean.csv"
recognition_sequence.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"识别序列已保存: {output_file}")
display(recognition_sequence.head(50))

识别序列已保存: D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007\recognition_sequence_clean.csv


,SOH,SOC,File,Time,Current_level,Zustand
2,96.2,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...,2024-09-05 14:46:43.380000+00:00,1.5,CHA
3,96.2,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...,2024-09-05 14:47:03.890000+00:00,0.0,PAUO
4,96.2,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...,2024-09-05 15:47:26.410000+00:00,-3.0,DCH
5,96.2,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...,2024-09-05 15:47:46.990000+00:00,0.0,PAUO
6,96.2,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...,2024-09-05 16:48:29.790000+00:00,3.0,CHA
7,96.2,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...,2024-09-05 16:48:50.370000+00:00,0.0,PAUO
8,96.2,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...,2024-09-05 21:45:11.350000+00:00,-1.5,DCH
9,96.2,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...,2024-09-05 21:45:31.920000+00:00,0.0,PAUO
10,96.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...,2024-09-05 22:45:54.520000+00:00,1.5,CHA
11,96.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...,2024-09-05 22:46:15.070000+00:00,0.0,PAUO
